# Round 11 | Worked feature example

Synthetic illustrative vectors only. Not a learned OTTO basis, not a score. This notebook is supplied unexecuted. It exercises the actual aggregation function on small explicit vectors without fitting a model.

In [ ]:
from pathlib import Path
import json, sys, importlib.util
ROOT = Path.home() / "otto_feature_round11"
assert ROOT.is_dir(), ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("otto_launcher_11", ROOT / "launch.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
def stage(name):
    return launcher.run_stage(name)
def require(relative, key, expected):
    value = json.loads((ROOT / relative).read_text())
    assert value[key] == expected, (relative, value.get(key))
    print(expected)
    return value
print("KERNEL_READY")
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected+plotly_mimetype"
from latent_features import affinity_summaries, names, normalize
vocabulary = np.array([10,20,30,40], dtype=np.int64)
base = normalize(np.array([[1.,0.],[0.,1.],[1.,1.],[-1.,0.]]))
query = np.broadcast_to(base, (3,4,2)).copy()
target = query.copy()
anchors = np.array([10,20], dtype=np.int64)
candidates = np.array([10,30,40,999], dtype=np.int64)
values = affinity_summaries(anchors,candidates,vocabulary,query,target)
print("SYNTHETIC EXAMPLE ONLY — explicit vectors, no learned project result")

## Signed affinities and missing vectors

In [ ]:
fig = go.Figure(go.Heatmap(z=values[:,:8], x=["latest","max","mean","std","weighted","centroid","latest-old","availability"], y=[str(x) for x in candidates]))
fig.update_layout(title="Synthetic click-view summaries — negative cosine retained", xaxis_title="Statistic", yaxis_title="Candidate ID (illustrative)")
fig.show()

## Candidate order is not a feature

In [ ]:
reversed_values = affinity_summaries(anchors,candidates[::-1],vocabulary,query,target)
np.testing.assert_allclose(reversed_values[::-1],values,atol=1e-6,rtol=1e-6)
fig = go.Figure(go.Bar(x=[str(x) for x in candidates],y=np.max(np.abs(reversed_values[::-1]-values),axis=1)))
fig.update_layout(title="Synthetic permutation check",yaxis_title="Maximum absolute difference",xaxis_title="Illustrative candidate")
fig.show()

## Missing/self-anchor conventions

In [ ]:
self_only = affinity_summaries(np.array([10],np.int64),np.array([10,999],np.int64),vocabulary,query,target)
np.testing.assert_array_equal(self_only,0)
fig=go.Figure(go.Bar(x=[str(x) for x in candidates],y=values[:,7]))
fig.update_layout(title="Synthetic available non-self anchor fraction",xaxis_title="Illustrative candidate",yaxis_title="Available fraction")
fig.show()

## Latest context versus older context

In [ ]:
changed = affinity_summaries(anchors[::-1],candidates,vocabulary,query,target)
fig=go.Figure()
fig.add_trace(go.Bar(name="Original anchor order",x=[str(x) for x in candidates],y=values[:,0]))
fig.add_trace(go.Bar(name="Reversed anchor order",x=[str(x) for x in candidates],y=changed[:,0]))
fig.update_layout(title="Synthetic query order carries information",xaxis_title="Illustrative candidate",yaxis_title="Latest cosine",barmode="group")
fig.show()